# 03 · Construct 14-day outcomes
Post-send outcomes per assignment, joined to Step 02 features → canonical `historical_feature_outcome` table.
**Eligible order:** same customer, same campaign, `completed`, `send_ts <= order_ts < send_ts + 14d`.
**NCM:** `net_revenue − cogs − shipping_cost − refund_amount_applicable` (net revenue already nets discounts and shipping revenue).

## Config — outcome window and return-timing rule (change here only)

In [1]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parent))
import numpy as np, pandas as pd
from src.common import TABLES, OUTCOME_DAYS, load_clean, load_processed, save
pd.set_option("display.width", 200); pd.set_option("display.max_columns", 30)
RETURN_RULE = "in_window"   # "in_window": refunds with return_ts < outcome_end_ts | "all_linked": every refund on outcome orders
TOL = 0.05

def applicable_refunds(outcome_orders, returns, rule=RETURN_RULE):
    """Link returns to eligible outcome orders by order_id and split refunds by timing relative to outcome_end_ts."""
    r = outcome_orders[["assignment_id", "order_id", "outcome_end_ts"]].merge(returns, on="order_id")
    r["in_window"] = r.return_ts < r.outcome_end_ts
    r["applicable"] = r.in_window if rule == "in_window" else True
    g = r.groupby("assignment_id")
    return pd.DataFrame({"refund_amount_applicable": r[r.applicable].groupby("assignment_id").refund_amount.sum(),
                         "in_window_return_count": r[r.in_window].groupby("assignment_id").size(),
                         "post_window_return_count": r[~r.in_window].groupby("assignment_id").size(),
                         "post_window_refund_amount": r[~r.in_window].groupby("assignment_id").refund_amount.sum()})

## Load inputs (Step 01 clean tables + Step 02 features)

In [2]:
t = load_clean(["historical_assignments", "orders", "returns", "offer_catalog"])
hf = load_processed("historical_features")
ha = t["historical_assignments"].assign(outcome_end_ts=lambda d: d.send_ts + pd.Timedelta(days=OUTCOME_DAYS))
orders, rets = t["orders"], t["returns"]
print(hf.shape, ha.shape)

(26600, 50) (26600, 8)


## Eligible outcome orders

In [3]:
m = ha.merge(orders, on=["customer_id", "campaign_id"], how="inner")
in_win = (m.order_ts >= m.send_ts) & (m.order_ts < m.outcome_end_ts)
oo = m[in_win & m.is_completed].copy()
assert oo.order_id.is_unique, "order counted for more than one assignment"
print(f"campaign-linked orders: {len(m)} | outside window: {(~in_win).sum()} | cancelled excluded: {(in_win & ~m.is_completed).sum()} | eligible: {len(oo)}")
print("organic (null campaign_id) orders excluded from outcomes:", orders.campaign_id.isna().sum())

campaign-linked orders: 3510 | outside window: 0 | cancelled excluded: 53 | eligible: 3457
organic (null campaign_id) orders excluded from outcomes: 3474


## FREE_SHIP realized promo cost — recover waived shipping fee from observed fee rule
Paid shipping appears only below a merchandise threshold. If the rule holds exactly on non-FREE_SHIP completed orders,
the waived fee on redeemed FREE_SHIP orders is inferred (flagged); otherwise it is left missing.

In [4]:
nf = orders[orders.is_completed & orders.offer_type_redeemed.ne("FREE_SHIP")]
FEE = nf.loc[nf.shipping_revenue > 0, "shipping_revenue"].unique()
THRESH = nf.loc[nf.shipping_revenue == 0, "merchandise_gross_revenue"].min()
paid_max = nf.loc[nf.shipping_revenue > 0, "merchandise_gross_revenue"].max()
rule_ok = len(FEE) == 1 and paid_max < THRESH
print(f"fee values: {FEE} | max gross w/ paid shipping: {paid_max} | min gross w/ free shipping: {THRESH} | rule holds: {rule_ok}")
fs = oo.offer_type_redeemed.eq("FREE_SHIP")
oo["waived_ship_fee"] = np.where(fs, np.where(oo.merchandise_gross_revenue < THRESH, FEE[0], 0.0) if rule_ok else np.nan, 0.0)
print("redeemed FREE_SHIP orders:", fs.sum(), "| with inferred waived fee > 0:", (oo.waived_ship_fee > 0).sum())

fee values: [5.99] | max gross w/ paid shipping: 74.99 | min gross w/ free shipping: 75.08 | rule holds: True
redeemed FREE_SHIP orders: 712 | with inferred waived fee > 0: 45


## Aggregate outcomes per assignment

In [5]:
oo["redeemed"] = oo.offer_redeemed.eq(1) & oo.offer_type_redeemed.eq(oo.offer_type)
oo["promo_cost"] = np.where(oo.redeemed, oo.discount_amount + oo.waived_ship_fee, 0.0)
g = oo.groupby("assignment_id")
agg = pd.DataFrame({"order_count_14d": g.size(), "gross_revenue_14d": g.merchandise_gross_revenue.sum(),
                    "discount_amount_14d": g.discount_amount.sum(), "shipping_revenue_14d": g.shipping_revenue.sum(),
                    "net_revenue_14d": g.net_revenue.sum(), "cogs_14d": g.cogs.sum(), "shipping_cost_14d": g.shipping_cost.sum(),
                    "offer_redeemed_14d": g.redeemed.max().astype(int), "promo_cost_14d": g.promo_cost.sum(min_count=1)})
out = ha[["assignment_id", "offer_type", "outcome_end_ts"]].set_index("assignment_id").join(agg).join(applicable_refunds(oo, rets))
fill0 = [c for c in out.columns if c not in ("offer_type", "outcome_end_ts", "promo_cost_14d")]
out[fill0] = out[fill0].fillna(0)
out["promo_cost_14d"] = out.promo_cost_14d.where(out.order_count_14d > 0, 0.0)
out["promo_cost_inferred"] = (out.offer_type.eq("FREE_SHIP") & out.offer_redeemed_14d.eq(1)).astype(int)
out["purchase_14d"] = (out.order_count_14d > 0).astype(int)
out["ncm_14d"] = out.net_revenue_14d - out.cogs_14d - out.shipping_cost_14d - out.refund_amount_applicable
out["ncm_14d_all_returns"] = out.ncm_14d - (out.post_window_refund_amount if RETURN_RULE == "in_window" else 0)
out["aov_14d"] = (out.net_revenue_14d / out.order_count_14d).where(out.order_count_14d > 0)
for c in ["order_count_14d", "in_window_return_count", "post_window_return_count"]:
    out[c] = out[c].astype(int)
OUTCOMES = ["outcome_end_ts", "purchase_14d", "order_count_14d", "gross_revenue_14d", "discount_amount_14d", "shipping_revenue_14d",
            "net_revenue_14d", "cogs_14d", "shipping_cost_14d", "offer_redeemed_14d", "refund_amount_applicable", "promo_cost_14d",
            "promo_cost_inferred", "ncm_14d", "aov_14d", "in_window_return_count", "post_window_return_count",
            "post_window_refund_amount", "ncm_14d_all_returns"]
ho = out[OUTCOMES].reset_index()

## Validate outcomes and financial reconciliation

In [6]:
alt = ho.gross_revenue_14d - ho.discount_amount_14d + ho.shipping_revenue_14d - ho.cogs_14d - ho.shipping_cost_14d - ho.refund_amount_applicable
checks = {
    "one row per assignment": ho.assignment_id.is_unique and len(ho) == len(ha),
    "purchase_14d binary, non-null": ho.purchase_14d.isin([0, 1]).all(),
    "ncm reconciles to gross-disc+ship_rev-cogs-ship_cost-refund": (ho.ncm_14d - alt).abs().max() < TOL * 5,
    "net revenue sums to eligible orders": abs(ho.net_revenue_14d.sum() - oo.net_revenue.sum()) < TOL,
    "order counts sum to eligible orders": ho.order_count_14d.sum() == len(oo),
    "refunds sum to linked returns": abs(ho.refund_amount_applicable.sum() + ho.post_window_refund_amount.sum()
                                         - rets[rets.order_id.isin(oo.order_id)].refund_amount.sum()) < TOL,
    "NO_OFFER has zero promo cost / redemption": (ho.merge(ha[["assignment_id", "offer_type"]]).query("offer_type=='NO_OFFER'")
                                                  [["promo_cost_14d", "offer_redeemed_14d"]].abs().sum().sum() == 0),
    "non-purchasers have zero financials": (ho.loc[ho.purchase_14d == 0, ["net_revenue_14d", "ncm_14d"]].abs().sum().sum() == 0),
}
print(pd.Series(checks).to_string()); assert all(checks.values())
print(f"max |ncm − alt| = {(ho.ncm_14d - alt).abs().max():.3f}")

one row per assignment                                         True
purchase_14d binary, non-null                                  True
ncm reconciles to gross-disc+ship_rev-cogs-ship_cost-refund    True
net revenue sums to eligible orders                            True
order counts sum to eligible orders                            True
refunds sum to linked returns                                  True
NO_OFFER has zero promo cost / redemption                      True
non-purchasers have zero financials                            True
max |ncm − alt| = 0.010


## Return timing counts

In [7]:
print(ho[["in_window_return_count", "post_window_return_count"]].sum().to_dict(),
      "| applicable refunds $", round(ho.refund_amount_applicable.sum(), 2), "| post-window refunds $", round(ho.post_window_refund_amount.sum(), 2))

{'in_window_return_count': 60, 'post_window_return_count': 309} | applicable refunds $ 15401.72 | post-window refunds $ 83775.98


## Assemble canonical historical feature-outcome table

In [8]:
hfo = hf.merge(ho, on="assignment_id", how="left", validate="1:1")
assert len(hfo) == len(hf) == len(ha) and hfo.purchase_14d.notna().all()
feature_cols = pd.read_csv(TABLES / "feature_schema.csv").query("family != 'context'").feature
assert not set(feature_cols) & set(OUTCOMES), "outcome field registered as feature"
print("historical_feature_outcome:", hfo.shape, "| purchase rate:", round(hfo.purchase_14d.mean(), 4))
hfo.groupby("campaign_id").agg(n=("purchase_14d", "size"), purchase_rate=("purchase_14d", "mean"), ncm_mean=("ncm_14d", "mean")).round(3).T

historical_feature_outcome: (26600, 69) | purchase rate: 0.13


campaign_id,CP01,CP02,CP03,CP04,CP05,CP06,CP07
n,3800.000,3800.000,3800.000,3800.000,3800.000,3800.000,3800.000
purchase_rate,0.131,0.131,0.127,0.130,0.122,0.138,0.132
ncm_mean,18.225,17.568,16.567,18.797,15.189,19.012,17.539


## Save outputs

In [9]:
save(ho, "historical_outcomes")
save(hfo, "historical_feature_outcome")
pd.DataFrame({"outcome": OUTCOMES}).to_csv(TABLES / "outcome_schema.csv", index=False)
print("saved historical_outcomes", ho.shape, "and historical_feature_outcome", hfo.shape)

saved historical_outcomes (26600, 20) and historical_feature_outcome (26600, 69)
